# Actividad 4: Métricas de Calidad de Resultados

**Estudiante:** Ovidio Alejandro Hernández Ruano  
**Matrícula:** A01796714  
**Fecha de entrega:** 14/06/2026

---

## Objetivo

Identificar métricas para la medición de la calidad de resultados derivados de la aplicación de modelos de aprendizaje supervisado y no supervisado, orientado al procesamiento de grandes volúmenes de datos, que permitan la selección de los modelos que mejor se ajusten a la tarea de aprendizaje a resolver.

## Dataset

**US Accidents (March 2023)** — registro de ~7.7 millones de accidentes de tráfico en EE.UU. (2016–2023).  
Fuente: Kaggle

## Particionamiento

Esta actividad retoma los criterios de particionamiento definidos en la Actividad 3 del Módulo 4:

| Variable | Valores | Descripción |
|---|---|---|
| `severity_group` | `low` (Severity 1–2) / `high` (Severity 3–4) | Discrimina la gravedad del accidente |
| `region_group` | `high_incidence` (CA, FL, TX) / `other` | Captura patrones geográficos |
| `weather_group` | `adverse` / `normal` | Refleja condiciones climáticas |

Las 3 variables de caracterización generan **8 particiones** (2³) denominadas P1–P8.

## Dependencias

In [1]:
import warnings
import builtins
import os
import math
from functools import reduce
warnings.filterwarnings('ignore')

os.environ.setdefault(
    "JAVA_HOME",
    "/opt/anaconda3/envs/env-pyspark/lib/jvm"
)

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, when, lit, count, avg, stddev, min as _min, max as _max,
    lower as _lower, percentile_approx
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (
    RandomForestClassifier, LogisticRegression, GBTClassifier
)
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import numpy as np
import pandas as pd
from datetime import datetime
import gdown

spark = SparkSession.builder \
    .appName("Actividad4_MetricasCalidad") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.maxResultSize", "1g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.storageFraction", "0.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark inicializado correctamente.")
print(f"Versión de Spark: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/13 20:53:23 WARN Utils: Your hostname, Ovidios-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.12 instead (on interface en0)
26/06/13 20:53:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 20:53:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark inicializado correctamente.
Versión de Spark: 4.1.2


---
## 1 Construcción de la muestra M

### 1.1 Justificación metodológica

La construcción de una muestra representativa sin sesgo es el fundamento de todo análisis de machine learning válido. Para este propósito se aplica **muestreo aleatorio estratificado proporcional** sobre las 8 particiones P1–P8 derivadas de las variables de caracterización.

#### Criterio anti-sesgo para el tamaño de cada partición Mi

A diferencia de la Actividad 3 (donde se usó una fracción fija del 2 % por partición), en esta actividad se aplica una **fracción de muestreo uniforme del 2 %** sobre cada Mi, lo cual permite los siguientes elementos:

1. **Proporcionalidad:** El tamaño de cada Mi en la muestra es proporcional a su peso en P → no se infla ni deflacta ningún estrato.
2. **Sin sesgo de selección:** Cada registro dentro de un estrato tiene **igual probabilidad** de ser seleccionado.
3. **Preservación de la distribución conjunta:** La distribución marginal y conjunta de las variables `severity_group`, `region_group` y `weather_group` en M refleja la de P.
4. **Reproducibilidad:** Se usa `seed=42` en todos los muestreos.


In [2]:
print("=" * 100)
print("1 Construcción de la muestra M")
print("=" * 100)

FILE_ID   = "1XRw4nXwOdNIuqr8nPnlNNUhYjEHAMzxN"
DATA_PATH = "/tmp/US_Accidents_March23.csv"

if not os.path.exists(DATA_PATH):
    print("Descargando dataset desde Google Drive...")
    gdown.download(id=FILE_ID, output=DATA_PATH, quiet=False)
    print("Descarga completada.")
else:
    print("Dataset ya existe localmente, omitiendo descarga.")

ml_columns = [
    "Severity", "State", "Weather_Condition",
    "Distance(mi)", "Temperature(F)", "Wind_Chill(F)",
    "Humidity(%)", "Pressure(in)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)"
]

raw_df = spark.read.csv(DATA_PATH, header=True, inferSchema=True).select(ml_columns)
raw_df = raw_df.dropna(subset=[c for c in ml_columns if c not in ["Weather_Condition"]])

total_P = raw_df.count()
print(f"\nPoblación P total: {total_P:,} registros")

1 Construcción de la muestra M
Descargando dataset desde Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1XRw4nXwOdNIuqr8nPnlNNUhYjEHAMzxN
From (redirected): https://drive.google.com/uc?id=1XRw4nXwOdNIuqr8nPnlNNUhYjEHAMzxN&confirm=t&uuid=7bbd0b5a-49a4-4944-a788-7bd835ae0811
To: /tmp/US_Accidents_March23.csv
100%|██████████| 3.06G/3.06G [02:19<00:00, 21.9MB/s]


Descarga completada.



Población P total: 5,246,811 registros


In [3]:
print("-" * 100)
print("\n Paso 1 — Variables de caracterización (criterios de particionamiento)")
print("-" * 100)

ADVERSE_CONDITIONS = [
    "rain", "snow", "fog", "storm", "thunder", "hail",
    "sleet", "freezing", "blizzard", "tornado", "wintry"
]
HIGH_INCIDENCE_STATES = ["CA", "FL", "TX"]

P_partitioned = raw_df \
    .withColumn(
        "severity_group",
        when(col("Severity").isin([1, 2]), "low").otherwise("high")
    ) \
    .withColumn(
        "region_group",
        when(col("State").isin(HIGH_INCIDENCE_STATES), "high_incidence").otherwise("other")
    ) \
    .withColumn(
        "weather_group",
        when(
            reduce(
                lambda a, b: a | b,
                [_lower(col("Weather_Condition")).contains(c) for c in ADVERSE_CONDITIONS]
            ),
            "adverse"
        ).otherwise("normal")
    ) \
    .withColumn(
        "severe_accident",
        when(col("Severity").isin([3, 4]), 1.0).otherwise(0.0)
    )

print("Variables de caracterización:")
print("   severity_group  : low (Severity 1–2)  / high (Severity 3–4)")
print("   region_group    : high_incidence (CA, FL, TX)  / other")
print("   weather_group   : adverse / normal")
print("   severe_accident : 0.0 (leve) / 1.0 (grave) — variable objetivo")

----------------------------------------------------------------------------------------------------

 Paso 1 — Variables de caracterización (criterios de particionamiento)
----------------------------------------------------------------------------------------------------
Variables de caracterización:
   severity_group  : low (Severity 1–2)  / high (Severity 3–4)
   region_group    : high_incidence (CA, FL, TX)  / other
   weather_group   : adverse / normal
   severe_accident : 0.0 (leve) / 1.0 (grave) — variable objetivo


In [4]:
print("-" * 100)
print("\n Paso 2 — Distribución de las 8 particiones en P")
print("-" * 100)

partition_dist = P_partitioned \
    .groupBy("severity_group", "region_group", "weather_group") \
    .count() \
    .withColumnRenamed("count", "n_registros") \
    .orderBy("severity_group", "region_group", "weather_group")

partition_rows = partition_dist.collect()

print(f"\n{'Partición':>5} | {'severity_group':>16} | {'region_group':>16} | {'weather_group':>14} | {'n_registros':>12} | {'% de P':>7}")
print("-" * 85)

partition_info = []
for i, row in enumerate(partition_rows, start=1):
    pct = row["n_registros"] / total_P * 100
    print(f"{'P'+str(i):>5} | {row['severity_group']:>16} | {row['region_group']:>16} | {row['weather_group']:>14} | {row['n_registros']:>12,} | {pct:>6.2f}%")
    partition_info.append({
        "name": f"P{i}",
        "severity_group": row["severity_group"],
        "region_group": row["region_group"],
        "weather_group": row["weather_group"],
        "n": row["n_registros"],
        "pct": pct
    })

print("-" * 85)
print(f"{'TOTAL':>5} | {'':>16} | {'':>16} | {'':>14} | {total_P:>12,} | {'100.00%':>7}")

----------------------------------------------------------------------------------------------------

 Paso 2 — Distribución de las 8 particiones en P
----------------------------------------------------------------------------------------------------



Partición |   severity_group |     region_group |  weather_group |  n_registros |  % de P
-------------------------------------------------------------------------------------
   P1 |             high |   high_incidence |        adverse |       20,243 |   0.39%
   P2 |             high |   high_incidence |         normal |      182,185 |   3.47%
   P3 |             high |            other |        adverse |       96,219 |   1.83%
   P4 |             high |            other |         normal |      413,876 |   7.89%
   P5 |              low |   high_incidence |        adverse |      164,674 |   3.14%
   P6 |              low |   high_incidence |         normal |    1,767,079 |  33.68%
   P7 |              low |            other |        adverse |      403,339 |   7.69%
   P8 |              low |            other |         normal |    2,199,196 |  41.91%
-------------------------------------------------------------------------------------
TOTAL |                  |                  |    

In [5]:
print("-" * 100)
print("\n Paso 3 — Muestreo aleatorio estratificado (fracción uniforme = 2%)")
print("-" * 100)

SAMPLE_FRACTION = 0.02   # 2 % por partición
SEED = 42

print(f"\nFracción de muestreo aplicada a cada partición Mi: {SAMPLE_FRACTION*100:.1f}%")
print(f"Seed para reproducibilidad: {SEED}")
print("\nRealizando muestreo estratificado...")

Mi_frames = []

for info in partition_info:
    partition_df = P_partitioned.filter(
        (col("severity_group") == info["severity_group"]) &
        (col("region_group")   == info["region_group"])   &
        (col("weather_group")  == info["weather_group"])
    )
    sample_i = partition_df.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SEED)
    Mi_frames.append(sample_i)

M = Mi_frames[0]
for frame in Mi_frames[1:]:
    M = M.union(frame)

M = M.cache()
total_M = M.count()

print(f"\n Muestra M construida exitosamente.")
print(f"   Registros en M: {total_M:,} ({total_M/total_P*100:.2f}% de P)")

----------------------------------------------------------------------------------------------------

 Paso 3 — Muestreo aleatorio estratificado (fracción uniforme = 2%)
----------------------------------------------------------------------------------------------------

Fracción de muestreo aplicada a cada partición Mi: 2.0%
Seed para reproducibilidad: 42

Realizando muestreo estratificado...



 Muestra M construida exitosamente.
   Registros en M: 105,914 (2.02% de P)


In [6]:
print("-" * 100)
print("\n Paso 4 — Verificación de representatividad de M")
print("-" * 100)

M_dist = M \
    .groupBy("severity_group", "region_group", "weather_group") \
    .count() \
    .withColumnRenamed("count", "n_M") \
    .orderBy("severity_group", "region_group", "weather_group") \
    .collect()

print(f"\n{'Partición':>5} | {'n en P':>10} | {'n en M':>8} | {'% P':>6} | {'% M':>6} | {'Δ (desviación)':>15} | {'Representativa?':>15}")
print("-" * 80)

max_deviation = 0.0
for i, (info, row) in enumerate(zip(partition_info, M_dist), start=1):
    pct_P = info["pct"]
    pct_M = row["n_M"] / total_M * 100
    delta  = abs(pct_P - pct_M)
    max_deviation = builtins.max(max_deviation, delta)
    ok = "✓ Sí" if delta < 0.5 else "⚠ Revisar"
    print(f"{'P'+str(i):>5} | {info['n']:>10,} | {row['n_M']:>8,} | {pct_P:>5.2f}% | {pct_M:>5.2f}% | {delta:>14.3f}% | {ok:>15}")

print("-" * 80)
print(f"Desviación máxima en la distribución de particiones: {max_deviation:.3f}%")
if max_deviation < 0.5:
    print("M es representativa de P — desviación < 0.5% en todos los estratos.")
else:
    print("Verificar estratos con desviación ≥ 0.5%.")

print(f"\n Distribución de la variable objetivo (severe_accident) en M:")
M.groupBy("severe_accident").count().orderBy("severe_accident").show()

----------------------------------------------------------------------------------------------------

 Paso 4 — Verificación de representatividad de M
----------------------------------------------------------------------------------------------------

Partición |     n en P |   n en M |    % P |    % M |  Δ (desviación) | Representativa?
--------------------------------------------------------------------------------
   P1 |     20,243 |      370 |  0.39% |  0.35% |          0.036% |            ✓ Sí
   P2 |    182,185 |    3,698 |  3.47% |  3.49% |          0.019% |            ✓ Sí
   P3 |     96,219 |    1,878 |  1.83% |  1.77% |          0.061% |            ✓ Sí
   P4 |    413,876 |    8,298 |  7.89% |  7.83% |          0.053% |            ✓ Sí
   P5 |    164,674 |    3,336 |  3.14% |  3.15% |          0.011% |            ✓ Sí
   P6 |  1,767,079 |   35,707 | 33.68% | 33.71% |          0.034% |            ✓ Sí
   P7 |    403,339 |    8,241 |  7.69% |  7.78% |          0.094% |       

In [7]:
print("-" * 100)
print("\n Paso 5 — Preprocesamiento de M")
print("-" * 100)

numeric_cols = [
    "Distance(mi)", "Temperature(F)", "Wind_Chill(F)",
    "Humidity(%)", "Pressure(in)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)"
]

M_cast = M
for c in numeric_cols:
    M_cast = M_cast.withColumn(c, col(c).cast(DoubleType()))

M_cast = M_cast.dropna(subset=numeric_cols)

def winsorize(df, column):
    q = df.approxQuantile(column, [0.25, 0.75], 0.05)
    Q1, Q3 = q[0], q[1]
    IQR     = Q3 - Q1
    lo, hi  = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return df.withColumn(
        column,
        when(col(column) < lo, lit(lo))
        .when(col(column) > hi, lit(hi))
        .otherwise(col(column))
    ), lo, hi

M_proc = M_cast
print("\nWinsorización por variable:")
for c in numeric_cols:
    M_proc, lo, hi = winsorize(M_proc, c)
    print(f"  {c:25s}: rango ajustado → [{lo:.3f}, {hi:.3f}]")

M_proc = M_proc.cache()
total_M_proc = M_proc.count()
print(f"\nRegistros tras preprocesamiento: {total_M_proc:,}")

print("\nEstadísticas descriptivas:")
M_proc.select(numeric_cols).describe().show(truncate=False)

----------------------------------------------------------------------------------------------------

 Paso 5 — Preprocesamiento de M
----------------------------------------------------------------------------------------------------

Winsorización por variable:
  Distance(mi)             : rango ajustado → [-0.831, 1.385]
  Temperature(F)           : rango ajustado → [5.000, 117.000]
  Wind_Chill(F)            : rango ajustado → [0.000, 120.000]
  Humidity(%)              : rango ajustado → [-5.500, 134.500]
  Pressure(in)             : rango ajustado → [27.970, 31.090]
  Visibility(mi)           : rango ajustado → [10.000, 10.000]
  Wind_Speed(mph)          : rango ajustado → [-7.500, 20.500]
  Precipitation(in)        : rango ajustado → [0.000, 0.000]

Registros tras preprocesamiento: 105,914

Estadísticas descriptivas:
+-------+-------------------+-----------------+------------------+------------------+------------------+--------------+-----------------+-----------------+
|summary

### 1.2 Resumen de la sección

| Aspecto | Detalle |
|---|---|
| Técnica | Muestreo Aleatorio Estratificado Proporcional |
| Fracción por estrato | 2 % uniforme (evita sesgo de sobre/infra-representación) |
| Particiones | 8 (combinación de severity_group × region_group × weather_group) |
| Seed | 42 (garantiza reproducibilidad) |
| Preprocesamiento | Casting de tipos + winsorización IQR (sin pérdida de registros) |
| Criterio anti-sesgo clave | La fracción **porcentual uniforme** preserva la distribución de P en M; una fracción **fija en absoluto** inflaría los estratos pequeños |

Con M = ∪ Mi se cumple que la distribución de patrones en M es estadísticamente equivalente a la de P, lo que garantiza que los modelos entrenados sobre M generalizarán a la población real.

---
## 2 Construcción Train – Test

### 2.1 Estrategia de particionamiento

El objetivo es dividir M en:
- **Conjunto de entrenamiento (Tr):** 80 % de M — usado para ajustar los parámetros del modelo.
- **Conjunto de prueba (Ts):** 20 % de M — reservado exclusivamente para la evaluación final.

**Condición de disjunción:** Tr ∩ Ts = ∅  
**Condición de cobertura:** Tr ∪ Ts = M

#### Por qué un split estratificado y no uno aleatorio simple

El dataset presenta un **desbalanceo de clases** (clase 0 ≈ 87%, clase 1 ≈ 13%). Con un split aleatorio simple (`randomSplit`) existe el riesgo estadístico de que Tr y Ts difieran en la proporción de la clase minoritaria, introduciendo un **sesgo de selección**.

La solución es el **split estratificado por clase**: se aplica `randomSplit([0.8, 0.2], seed=42)` de forma independiente para cada clase, asegurando que ambos subconjuntos mantengan la misma distribución de `severe_accident` que M.

**Porcentaje de división (80/20):**
- Con M ≈ 150k–200k registros, el 80% proporciona al modelo suficientes instancias de la clase minoritaria para aprender sus patrones.
- El 20% de prueba es suficientemente grande para obtener estimaciones estables de las métricas.

In [8]:
print("=" * 100)
print("2 Construcción Train – Test")
print("=" * 100)

TRAIN_RATIO = 0.80
TEST_RATIO  = 0.20
SEED        = 42

print(f"\nConfiguración:")
print(f"  Ratio entrenamiento : {TRAIN_RATIO * 100:.0f}%")
print(f"  Ratio prueba        : {TEST_RATIO  * 100:.0f}%")
print(f"  Seed                : {SEED}")
print(f"  Técnica             : Split estratificado por clase (severe_accident)")

class_0 = M_proc.filter(col("severe_accident") == 0.0)
class_1 = M_proc.filter(col("severe_accident") == 1.0)

train_0, test_0 = class_0.randomSplit([TRAIN_RATIO, TEST_RATIO], seed=SEED)
train_1, test_1 = class_1.randomSplit([TRAIN_RATIO, TEST_RATIO], seed=SEED)

train_df = train_0.union(train_1).cache()
test_df  = test_0.union(test_1).cache()

n_train = train_df.count()
n_test  = test_df.count()
n_total = n_train + n_test

print(f"\n Resultados del split:")
print(f"   Total M procesado    : {n_total:,}")
print(f"   Entrenamiento (Tr)   : {n_train:,} ({n_train/n_total*100:.1f}%)")
print(f"   Prueba        (Ts)   : {n_test:,}  ({n_test/n_total*100:.1f}%)")

2 Construcción Train – Test

Configuración:
  Ratio entrenamiento : 80%
  Ratio prueba        : 20%
  Seed                : 42
  Técnica             : Split estratificado por clase (severe_accident)

 Resultados del split:
   Total M procesado    : 105,914
   Entrenamiento (Tr)   : 84,589 (79.9%)
   Prueba        (Ts)   : 21,325  (20.1%)


In [9]:
print("-" * 100)
print("\n Verificación de distribución de clases en Tr y Ts")
print("-" * 100)

def class_distribution(df, name):
    rows = df.groupBy("severe_accident").count().orderBy("severe_accident").collect()
    total = builtins.sum(r["count"] for r in rows)
    print(f"\n{name}:")
    for r in rows:
        label = "Leve (0) " if int(r["severe_accident"]) == 0 else "Grave (1)"
        pct = r["count"] / total * 100
        print(f"  Clase {int(r['severe_accident'])} ({label}): {r['count']:>8,} ({pct:.2f}%)")
    return {int(r["severe_accident"]): r["count"] / total for r in rows}

dist_M  = class_distribution(M_proc, "M (completo)")
dist_tr = class_distribution(train_df, "Tr (entrenamiento)")
dist_ts = class_distribution(test_df,  "Ts (prueba)")

print("\n Comparación de proporciones de clase:")
print(f"  {'Conjunto':>20} | {'Clase 0 (Leve)':>16} | {'Clase 1 (Grave)':>16}")
print(f"  {'-'*60}")
print(f"  {'M':>20} | {dist_M.get(0,0)*100:>15.2f}% | {dist_M.get(1,0)*100:>15.2f}%")
print(f"  {'Tr (entrenamiento)':>20} | {dist_tr.get(0,0)*100:>15.2f}% | {dist_tr.get(1,0)*100:>15.2f}%")
print(f"  {'Ts (prueba)':>20} | {dist_ts.get(0,0)*100:>15.2f}% | {dist_ts.get(1,0)*100:>15.2f}%")

delta_cl0 = abs(dist_tr.get(0, 0) - dist_ts.get(0, 0)) * 100
delta_cl1 = abs(dist_tr.get(1, 0) - dist_ts.get(1, 0)) * 100
print(f"\n  Δ entre Tr y Ts — Clase 0: {delta_cl0:.3f}%  |  Clase 1: {delta_cl1:.3f}%")
if delta_cl0 < 0.5 and delta_cl1 < 0.5:
    print("Distribución de clases equivalente entre Tr y Ts — sin sesgo detectado.")
else:
    print("Verificar distribución de clases.")

----------------------------------------------------------------------------------------------------

 Verificación de distribución de clases en Tr y Ts
----------------------------------------------------------------------------------------------------

M (completo):
  Clase 0 (Leve (0) ):   91,670 (86.55%)
  Clase 1 (Grave (1)):   14,244 (13.45%)

Tr (entrenamiento):
  Clase 0 (Leve (0) ):   73,280 (86.63%)
  Clase 1 (Grave (1)):   11,309 (13.37%)

Ts (prueba):
  Clase 0 (Leve (0) ):   18,390 (86.24%)
  Clase 1 (Grave (1)):    2,935 (13.76%)

 Comparación de proporciones de clase:
              Conjunto |   Clase 0 (Leve) |  Clase 1 (Grave)
  ------------------------------------------------------------
                     M |           86.55% |           13.45%
    Tr (entrenamiento) |           86.63% |           13.37%
           Ts (prueba) |           86.24% |           13.76%

  Δ entre Tr y Ts — Clase 0: 0.394%  |  Clase 1: 0.394%
Distribución de clases equivalente entre Tr y 

In [10]:
print("-" * 100)
print("\n Verificación de disjunción: Tr ∩ Ts = ∅")
print("-" * 100)

print("""\n Disjunción:
  La función randomSplit() de PySpark divide el RDD subyacente asignando cada registro a
  exactamente un subconjunto según su hash interno y la semilla especificada.
  No existe un mecanismo de muestreo con reemplazo entre particiones (a menos
  que se use withReplacement=True explícitamente).  
  
  Al aplicar randomSplit de forma independiente sobre class_0 y class_1 y luego
  unir los resultados, cada registro de M aparece en Tr O en Ts, nunca en ambos.
  
  Verificación adicional: n(Tr) + n(Ts) == n(M)
""")

assert n_train + n_test == n_total, \
    f"Error: {n_train} + {n_test} ≠ {n_total}"
print(f"  {n_train:,} + {n_test:,} = {n_total:,}  ---  Tr ∪ Ts = M  y  Tr ∩ Ts = ∅")

----------------------------------------------------------------------------------------------------

 Verificación de disjunción: Tr ∩ Ts = ∅
----------------------------------------------------------------------------------------------------

 Disjunción:
  La función randomSplit() de PySpark divide el RDD subyacente asignando cada registro a
  exactamente un subconjunto según su hash interno y la semilla especificada.
  No existe un mecanismo de muestreo con reemplazo entre particiones (a menos
  que se use withReplacement=True explícitamente).  

  Al aplicar randomSplit de forma independiente sobre class_0 y class_1 y luego
  unir los resultados, cada registro de M aparece en Tr O en Ts, nunca en ambos.

  Verificación adicional: n(Tr) + n(Ts) == n(M)

  84,589 + 21,325 = 105,914  ---  Tr ∪ Ts = M  y  Tr ∩ Ts = ∅


In [11]:
print("-" * 100)
print("\n Cálculo de pesos de clase para corregir desbalanceo")
print("-" * 100)

class_counts_train = train_df.groupBy("severe_accident").count().collect()
n_total_train = builtins.sum(r["count"] for r in class_counts_train)
n_classes     = len(class_counts_train)

class_weights = {
    int(r["severe_accident"]): n_total_train / (n_classes * r["count"])
    for r in class_counts_train
}

print("\nFórmula: w(k) = N_total / (N_clases × N_k)")
for cls, w in sorted(class_weights.items()):
    label = "Leve (0) " if cls == 0 else "Grave (1)"
    print(f"  Clase {cls} ({label}): peso = {w:.4f}")

train_weighted = train_df.withColumn(
    "class_weight",
    when(col("severe_accident") == 1.0, lit(class_weights[1]))
    .otherwise(lit(class_weights[0]))
)
test_weighted = test_df.withColumn(
    "class_weight",
    when(col("severe_accident") == 1.0, lit(class_weights[1]))
    .otherwise(lit(class_weights[0]))
)

print("\nPesos aplicados a train_weighted y test_weighted.")

----------------------------------------------------------------------------------------------------

 Cálculo de pesos de clase para corregir desbalanceo
----------------------------------------------------------------------------------------------------

Fórmula: w(k) = N_total / (N_clases × N_k)
  Clase 0 (Leve (0) ): peso = 0.5772
  Clase 1 (Grave (1)): peso = 3.7399

Pesos aplicados a train_weighted y test_weighted.


### 2.2 Resumen de la sección

| Aspecto | Detalle |
|---|---|
| Técnica | Split estratificado por clase (Tr 80 % / Ts 20 %) |
| Condición Tr ∩ Ts = ∅ | Garantizada por el mecanismo de `randomSplit` de PySpark |
| Condición Tr ∪ Ts = M | Verificada mediante igualdad de conteos |
| Anti-sesgo de clase | La misma proporción de clase 0/1 en Tr y Ts → Δ < 0.5% |
| Corrección de desbalanceo | Pesos inversamente proporcionales a la frecuencia de clase |

---
## 3 Selección de métricas para medir calidad de resultados

### 3.1 Contexto y consideraciones para Big Data

Al trabajar con grandes volúmenes de datos en un entorno distribuido (PySpark), la selección de métricas debe considerar:

1. **Escalabilidad:** Las métricas deben poder calcularse de forma distribuida, evitando operaciones que requieran recolectar todos los datos en el driver.
2. **Desbalanceo de clases:** Un dataset desbalanceado (~87%/13%) invalida el uso de *Accuracy* como métrica principal, ya que un modelo trivial que predice siempre la clase mayoritaria obtendría ~87% sin aprender nada.
3. **Relevancia del dominio:** En detección de accidentes graves, el costo de un **falso negativo** (no detectar un accidente grave) es mucho mayor que el de un **falso positivo**, lo que favorece métricas sensibles al Recall.

### 3.2 Métricas para Aprendizaje Supervisado (Clasificación Binaria)

#### Métricas de la Matriz de Confusión

| Métrica | Fórmula | Descripción |
|---|---|---|
| **Accuracy** | (TP+TN)/(TP+TN+FP+FN) | Línea base, pero insuficiente con datos desbalanceados |
| **Precision** | TP/(TP+FP) | ¿De los accidentes predichos como graves, cuántos lo son realmente? |
| **Recall (Sensibilidad)** | TP/(TP+FN) | **Métrica principal:** ¿De todos los accidentes graves, cuántos detectamos? |
| **F1-Score** | 2·Precision·Recall/(P+R) | Balance armónico entre Precision y Recall — clave con clases desbalanceadas |
| **F2-Score** | 5·P·R/(4P+R) | Penaliza más los FN que los FP — apropiado en seguridad vial |
| **MCC** | (TP·TN−FP·FN)/√(...) | Métrica robusta a desbalanceo, considerada la mejor métrica single-value |

#### Métricas de Curvas

| Métrica | Descripción |
|---|---|
| **AUC-ROC** | Área bajo la curva ROC — mide la capacidad discriminativa del modelo en todos los umbrales; robusta a desbalanceo moderado |
| **AUC-PR** | Área bajo la curva Precision-Recall — **más informativa que AUC-ROC** con clases muy desbalanceadas, ya que no usa TN (que son abundantes y fáciles) |

### 3.3 Métricas para Aprendizaje No Supervisado (Clustering)

| Métrica | Rango | Interpretación | Descripción |
|---|---|---|---|
| **Silhouette Score** | [-1, 1] | Más cercano a 1 = mejor | Mide cohesión intra-cluster y separación inter-cluster; implementado nativamente en PySpark |
| **WCSS (Inercia)** | [0, ∞) | Menor = mejor | Suma de distancias al centroide — criterio del "codo" para seleccionar k |
| **Davies-Bouldin Index** | [0, ∞) | Menor = mejor | Ratio entre dispersión interna y separación entre clusters; valores < 1 indican buen agrupamiento |
| **Calinski-Harabasz Score** | [0, ∞) | Mayor = mejor | Ratio varianza inter/intra-cluster; escalable a Big Data |

### 3.4 Métricas seleccionadas para experimentación

**Supervisado:** AUC-ROC, F1-Score, Recall, Precision, Accuracy, MCC  
**No supervisado:** Silhouette Score, WCSS, Davies-Bouldin Index

**Métrica primaria (supervisado):** AUC-ROC + F1-Score (clase 1)  
**Métrica primaria (clustering):** Silhouette Score

In [12]:
print("=" * 100)
print("3 Selección de métricas para medir calidad de resultados")
print("=" * 100)

def compute_supervised_metrics(predictions_df, label_col="severe_accident", pred_col="prediction", prob_col=None):
    """
    Calcula métricas completas para clasificación binaria.
    Retorna un dict con todas las métricas.
    """
    TP = predictions_df.filter((col(label_col) == 1) & (col(pred_col) == 1)).count()
    TN = predictions_df.filter((col(label_col) == 0) & (col(pred_col) == 0)).count()
    FP = predictions_df.filter((col(label_col) == 0) & (col(pred_col) == 1)).count()
    FN = predictions_df.filter((col(label_col) == 1) & (col(pred_col) == 0)).count()

    accuracy  = (TP + TN) / (TP + TN + FP + FN) if (TP+TN+FP+FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    f2        = 5 * precision * recall / (4 * precision + recall) if (4 * precision + recall) > 0 else 0.0

    mcc_num   = (TP * TN) - (FP * FN)
    mcc_den   = math.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN)) if (TP+FP)*(TP+FN)*(TN+FP)*(TN+FN) > 0 else 1.0
    mcc       = mcc_num / mcc_den

    auc_roc = None
    auc_pr  = None
    if prob_col is not None:
        bin_eval = BinaryClassificationEvaluator(
            labelCol=label_col, rawPredictionCol="rawPrediction"
        )
        bin_eval.setMetricName("areaUnderROC")
        auc_roc = bin_eval.evaluate(predictions_df)
        bin_eval.setMetricName("areaUnderPR")
        auc_pr  = bin_eval.evaluate(predictions_df)

    return {
        "TP": TP, "TN": TN, "FP": FP, "FN": FN,
        "Accuracy":  accuracy,
        "Precision": precision,
        "Recall":    recall,
        "F1-Score":  f1,
        "F2-Score":  f2,
        "MCC":       mcc,
        "AUC-ROC":   auc_roc,
        "AUC-PR":    auc_pr
    }


def print_supervised_metrics(metrics, model_name):
    """Imprime tabla formateada de métricas supervisadas."""
    print(f"\n Métricas — {model_name}")
    print("-" * 50)
    print(f"  {'Matriz de confusión':}")
    print(f"    TP = {metrics['TP']:,}  |  FP = {metrics['FP']:,}")
    print(f"    FN = {metrics['FN']:,}  |  TN = {metrics['TN']:,}")
    print()
    for k in ["Accuracy", "Precision", "Recall", "F1-Score", "F2-Score", "MCC"]:
        print(f"  {k:15s}: {metrics[k]:.4f}")
    if metrics["AUC-ROC"] is not None:
        print(f"  {'AUC-ROC':15s}: {metrics['AUC-ROC']:.4f}")
        print(f"  {'AUC-PR':15s}: {metrics['AUC-PR']:.4f}")


def compute_davies_bouldin(predictions_df, cluster_col="prediction", feature_col="scaledFeatures"):
    """
    Calcula el Davies-Bouldin Index vía Pandas (muestra) dado el volumen manejable de centroides.
    """
    cluster_stats_local = predictions_df \
        .groupBy(cluster_col) \
        .agg(
            avg("Distance(mi)").alias("c_dist"),
            avg("Temperature(F)").alias("c_temp"),
            avg("Humidity(%)").alias("c_hum"),
            avg("Visibility(mi)").alias("c_vis"),
            avg("Wind_Speed(mph)").alias("c_wind"),
            stddev("Distance(mi)").alias("s_dist"),
            stddev("Temperature(F)").alias("s_temp"),
            count("*").alias("n")
        ).toPandas()

    k = len(cluster_stats_local)
    centroids = cluster_stats_local[["c_dist", "c_temp", "c_hum", "c_vis", "c_wind"]].values
    avg_intra = ((cluster_stats_local["s_dist"].fillna(0) +
                  cluster_stats_local["s_temp"].fillna(0)) / 2).values

    db = 0.0
    for i in range(k):
        max_ratio = 0.0
        for j in range(k):
            if i == j:
                continue
            dist_ij = np.linalg.norm(centroids[i] - centroids[j])
            if dist_ij > 0:
                ratio = (avg_intra[i] + avg_intra[j]) / dist_ij
                max_ratio = builtins.max(max_ratio, ratio)
        db += max_ratio

    return db / k


print("\nFunciones de evaluación definidas:")
print("  compute_supervised_metrics()  — Accuracy, Precision, Recall, F1, F2, MCC, AUC-ROC, AUC-PR")
print("  print_supervised_metrics()    — visualización formateada")
print("  compute_davies_bouldin()      — Davies-Bouldin Index para clustering")

3 Selección de métricas para medir calidad de resultados

Funciones de evaluación definidas:
  compute_supervised_metrics()  — Accuracy, Precision, Recall, F1, F2, MCC, AUC-ROC, AUC-PR
  print_supervised_metrics()    — visualización formateada
  compute_davies_bouldin()      — Davies-Bouldin Index para clustering


---
## 4 Entrenamiento de Modelos de Aprendizaje

### 4.1 Estrategia general de entrenamiento

Para garantizar la calidad de los modelos se aplican las siguientes técnicas:

| Técnica | Propósito |
|---|---|
| **Pesos de clase** | Corrige el desbalanceo ~6:1 sin sobremuestrear |
| **Pipeline de preprocesamiento** | Encapsula VectorAssembler + StandardScaler → evita fuga de datos (data leakage) |
| **CrossValidator (3-fold)** | Búsqueda de hiperparámetros con validación cruzada → previene overfitting |
| **Grid de hiperparámetros** | Se evalúan combinaciones de parámetros clave de cada algoritmo |
| **Métrica de optimización** | AUC-ROC (robusta a desbalanceo) |
| **Conjunto de prueba reservado** | Ts nunca se usa durante el entrenamiento — evaluación final sin sesgo |

### 4.2 Modelos a entrenar

**Supervisados:**
- **Random Forest Classifier** — robusto, escalable en Spark, maneja no-linealidades.
- **Logistic Regression** — modelo lineal de referencia (baseline); interpretable.
- **Gradient Boosted Trees (GBT)** — boosting secuencial, generalmente superior en clasificación tabular.

**No supervisado:**
- **K-Means Clustering** — identificación de grupos de condiciones climáticas en accidentes.

In [13]:
print("=" * 100)
print("4 Entrenamiento de Modelos de Aprendizaje")
print("=" * 100)

FEATURE_COLS = [
    "Distance(mi)", "Temperature(F)", "Wind_Chill(F)",
    "Humidity(%)", "Pressure(in)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)"
]
TARGET_COL = "severe_accident"

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="features",
    handleInvalid="skip"
)
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withMean=True,
    withStd=True
)

print(f"Características de entrada ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Variable objetivo: {TARGET_COL}")
print(f"Corrección de desbalanceo: clase 0 --- peso {class_weights[0]:.4f}, clase 1 --- peso {class_weights[1]:.4f}")

4 Entrenamiento de Modelos de Aprendizaje
Características de entrada (8): ['Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']
Variable objetivo: severe_accident
Corrección de desbalanceo: clase 0 --- peso 0.5772, clase 1 --- peso 3.7399


In [14]:
print("-" * 100)
print("\n Modelo 1: Random Forest Classifier con CrossValidator")
print("-" * 100)

rf = RandomForestClassifier(
    labelCol=TARGET_COL,
    featuresCol="scaledFeatures",
    weightCol="class_weight",
    seed=42
)

rf_pipeline = Pipeline(stages=[assembler, scaler, rf])

rf_param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees,        [50, 100]) \
    .addGrid(rf.maxDepth,        [8]) \
    .addGrid(rf.subsamplingRate, [0.8]) \
    .build()

rf_evaluator = BinaryClassificationEvaluator(
    labelCol=TARGET_COL,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

rf_cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_param_grid,
    evaluator=rf_evaluator,
    numFolds=2,
    seed=42,
    parallelism=1
)

print(f"  Grid de búsqueda: {len(rf_param_grid)} combinaciones × 2 folds = {len(rf_param_grid)*2} modelos")
print("  Métrica de optimización: AUC-ROC")
print("  Entrenando...")

t0_rf = datetime.now()
rf_cv_model = rf_cv.fit(train_weighted)
t1_rf = datetime.now()

print(f"  Entrenamiento completado en {(t1_rf - t0_rf).total_seconds():.1f}s")

best_rf_pipeline = rf_cv_model.bestModel
best_rf = best_rf_pipeline.stages[-1]

print(f"\n  Mejores hiperparámetros encontrados:")
print(f"    numTrees        : {best_rf.getNumTrees}")
print(f"    maxDepth        : {best_rf.getOrDefault('maxDepth')}")
print(f"    subsamplingRate : {best_rf.getOrDefault('subsamplingRate')}")

avg_metrics_rf = rf_cv_model.avgMetrics
print(f"\n  AUC-ROC promedio por combinación (validación cruzada):")
for i, score in enumerate(avg_metrics_rf):
    print(f"    Combinación {i+1}: {score:.4f}")
print(f"  Mejor AUC-ROC CV: {builtins.max(avg_metrics_rf):.4f}")

----------------------------------------------------------------------------------------------------

 Modelo 1: Random Forest Classifier con CrossValidator
----------------------------------------------------------------------------------------------------
  Grid de búsqueda: 2 combinaciones × 2 folds = 4 modelos
  Métrica de optimización: AUC-ROC
  Entrenando...


  Entrenamiento completado en 124.1s

  Mejores hiperparámetros encontrados:
    numTrees        : 50
    maxDepth        : 8
    subsamplingRate : 0.8

  AUC-ROC promedio por combinación (validación cruzada):
    Combinación 1: 0.7429
    Combinación 2: 0.7429
  Mejor AUC-ROC CV: 0.7429


In [15]:
print("-" * 100)
print("\n Evaluación de Random Forest en Ts (conjunto de prueba)")
print("-" * 100)

rf_test_preds = rf_cv_model.transform(test_weighted)
rf_metrics    = compute_supervised_metrics(rf_test_preds, prob_col="probability")
print_supervised_metrics(rf_metrics, "Random Forest")

feat_imp = best_rf.featureImportances.toArray()
imp_df   = pd.DataFrame({"Feature": FEATURE_COLS, "Importance": feat_imp}) \
             .sort_values("Importance", ascending=False)

print("\n  Importancia de características (RF):")
for _, row in imp_df.iterrows():
    bar = chr(9608) * int(row["Importance"] * 50)
    print(f"    {row['Feature']:22s} {bar:50s} {row['Importance']:.4f}")

----------------------------------------------------------------------------------------------------

 Evaluación de Random Forest en Ts (conjunto de prueba)
----------------------------------------------------------------------------------------------------

 Métricas — Random Forest
--------------------------------------------------
  Matriz de confusión
    TP = 2,080  |  FP = 5,565
    FN = 855  |  TN = 12,825

  Accuracy       : 0.6989
  Precision      : 0.2721
  Recall         : 0.7087
  F1-Score       : 0.3932
  F2-Score       : 0.5365
  MCC            : 0.2917
  AUC-ROC        : 0.7510
  AUC-PR         : 0.2924

  Importancia de características (RF):
    Distance(mi)           ███████████████████████████████████████            0.7971
    Pressure(in)           ███                                                0.0654
    Humidity(%)            █                                                  0.0386
    Wind_Speed(mph)        █                                                  

In [16]:
print("-" * 100)
print("\n Modelo 2: Logistic Regression con modelo baseline")
print("-" * 100)

lr = LogisticRegression(
    labelCol=TARGET_COL,
    featuresCol="scaledFeatures",
    weightCol="class_weight",
    maxIter=100,
    elasticNetParam=0.5
)

lr_pipeline = Pipeline(stages=[assembler, scaler, lr])

lr_param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam,       [0.01, 0.1]) \
    .addGrid(lr.elasticNetParam,[0.5]) \
    .build()

lr_evaluator = BinaryClassificationEvaluator(
    labelCol=TARGET_COL, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

lr_cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=lr_param_grid,
    evaluator=lr_evaluator,
    numFolds=2,
    seed=42,
    parallelism=1
)

print(f"  Grid de búsqueda: {len(lr_param_grid)} combinaciones × 2 folds")
print("  Regularización: ElasticNet (L1 + L2) para prevenir overfitting")
print("  Entrenando...")

t0_lr = datetime.now()
lr_cv_model = lr_cv.fit(train_weighted)
t1_lr = datetime.now()

print(f"  Entrenamiento completado en {(t1_lr - t0_lr).total_seconds():.1f}s")

best_lr = lr_cv_model.bestModel.stages[-1]
print(f"\n  Mejores hiperparámetros:")
print(f"    regParam        : {best_lr.getRegParam()}")
print(f"    elasticNetParam : {best_lr.getElasticNetParam()}")
print(f"  Mejor AUC-ROC CV: {builtins.max(lr_cv_model.avgMetrics):.4f}")

lr_test_preds = lr_cv_model.transform(test_weighted)
lr_metrics    = compute_supervised_metrics(lr_test_preds, prob_col="probability")
print_supervised_metrics(lr_metrics, "Logistic Regression")

----------------------------------------------------------------------------------------------------

 Modelo 2: Logistic Regression con modelo baseline
----------------------------------------------------------------------------------------------------
  Grid de búsqueda: 2 combinaciones × 2 folds
  Regularización: ElasticNet (L1 + L2) para prevenir overfitting
  Entrenando...


  Entrenamiento completado en 17.3s

  Mejores hiperparámetros:
    regParam        : 0.1
    elasticNetParam : 0.5
  Mejor AUC-ROC CV: 0.6653

 Métricas — Logistic Regression
--------------------------------------------------
  Matriz de confusión
    TP = 2,286  |  FP = 11,651
    FN = 649  |  TN = 6,739

  Accuracy       : 0.4232
  Precision      : 0.1640
  Recall         : 0.7789
  F1-Score       : 0.2710
  F2-Score       : 0.4451
  MCC            : 0.1052
  AUC-ROC        : 0.6719
  AUC-PR         : 0.2384


In [17]:
print("-" * 100)
print("\n Modelo 3: GBT - Gradient Boosted Trees")
print("-" * 100)

gbt = GBTClassifier(
    labelCol=TARGET_COL,
    featuresCol="scaledFeatures",
    seed=42,
    subsamplingRate=0.8
)

gbt_pipeline = Pipeline(stages=[assembler, scaler, gbt])

gbt_param_grid = ParamGridBuilder() \
    .addGrid(gbt.maxIter,  [20, 50]) \
    .addGrid(gbt.maxDepth, [5]) \
    .addGrid(gbt.stepSize, [0.1]) \
    .build()

gbt_evaluator = BinaryClassificationEvaluator(
    labelCol=TARGET_COL, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

gbt_cv = CrossValidator(
    estimator=gbt_pipeline,
    estimatorParamMaps=gbt_param_grid,
    evaluator=gbt_evaluator,
    numFolds=2,
    seed=42,
    parallelism=1
)

print(f"  Grid de búsqueda: {len(gbt_param_grid)} combinaciones × 2 folds")
print("  subsamplingRate=0.8 para prevenir overfitting (stochastic boosting)")
print("  Entrenando...")

t0_gbt = datetime.now()
gbt_cv_model = gbt_cv.fit(train_df)
t1_gbt = datetime.now()

print(f"  Entrenamiento completado en {(t1_gbt - t0_gbt).total_seconds():.1f}s")

best_gbt = gbt_cv_model.bestModel.stages[-1]
print(f"\n  Mejores hiperparámetros:")
print(f"    maxIter   : {best_gbt.getMaxIter()}")
print(f"    maxDepth  : {best_gbt.getMaxDepth()}")
print(f"    stepSize  : {best_gbt.getStepSize()}")
print(f"  Mejor AUC-ROC CV: {builtins.max(gbt_cv_model.avgMetrics):.4f}")

gbt_test_preds = gbt_cv_model.transform(test_df)
gbt_metrics    = compute_supervised_metrics(gbt_test_preds, prob_col="probability")
print_supervised_metrics(gbt_metrics, "Gradient Boosted Trees")

----------------------------------------------------------------------------------------------------

 Modelo 3: GBT - Gradient Boosted Trees
----------------------------------------------------------------------------------------------------
  Grid de búsqueda: 2 combinaciones × 2 folds
  subsamplingRate=0.8 para prevenir overfitting (stochastic boosting)
  Entrenando...


  Entrenamiento completado en 287.2s

  Mejores hiperparámetros:
    maxIter   : 50
    maxDepth  : 5
    stepSize  : 0.1
  Mejor AUC-ROC CV: 0.7433

 Métricas — Gradient Boosted Trees
--------------------------------------------------
  Matriz de confusión
    TP = 11  |  FP = 14
    FN = 2,924  |  TN = 18,376

  Accuracy       : 0.8622
  Precision      : 0.4400
  Recall         : 0.0037
  F1-Score       : 0.0074
  F2-Score       : 0.0047
  MCC            : 0.0301
  AUC-ROC        : 0.7542
  AUC-PR         : 0.3004


In [18]:
print("-" * 100)
print("\n Modelo 4: K-Means Clustering (aprendizaje no supervisado)")
print("-" * 100)

assembler_km = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features", handleInvalid="skip")
scaler_km    = StandardScaler(inputCol="features", outputCol="scaledFeatures", withMean=True, withStd=True)

preproc_pipeline = Pipeline(stages=[assembler_km, scaler_km])
preproc_model    = preproc_pipeline.fit(M_proc)
M_scaled         = preproc_model.transform(M_proc).cache()
M_scaled.count()

print("\n  Búsqueda del k óptimo (Silhouette Score, muestra=15%)...")

sample_scaled = M_scaled.sample(False, 0.15, seed=42).cache()
sample_scaled.count()

silhouette_scores = {}
wcss_values       = {}

km_evaluator = ClusteringEvaluator(
    predictionCol="prediction",
    featuresCol="scaledFeatures",
    metricName="silhouette"
)

for k in range(2, 7):
    km_temp = KMeans(k=k, seed=42, maxIter=15, initMode="k-means||", featuresCol="scaledFeatures")
    km_mod  = km_temp.fit(sample_scaled)
    km_pred = km_mod.transform(sample_scaled)
    sil     = km_evaluator.evaluate(km_pred)
    wcss    = km_mod.summary.trainingCost
    silhouette_scores[k] = sil
    wcss_values[k]       = wcss
    print(f"    k={k}: Silhouette={sil:.4f}  WCSS={wcss:,.0f}")

optimal_k = builtins.max(silhouette_scores, key=silhouette_scores.get)
print(f"\n  k óptimo seleccionado: {optimal_k}  (Silhouette={silhouette_scores[optimal_k]:.4f})")

----------------------------------------------------------------------------------------------------

 Modelo 4: K-Means Clustering (aprendizaje no supervisado)
----------------------------------------------------------------------------------------------------

  Búsqueda del k óptimo (Silhouette Score, muestra=15%)...
    k=2: Silhouette=0.3791  WCSS=71,901
    k=3: Silhouette=0.3043  WCSS=61,402
    k=4: Silhouette=0.3280  WCSS=53,621
    k=5: Silhouette=0.2684  WCSS=51,765
    k=6: Silhouette=0.2916  WCSS=45,178

  k óptimo seleccionado: 2  (Silhouette=0.3791)


In [19]:
print("-" * 100)
print(f"\n  Entrenamiento final de K-Means (k={optimal_k}) sobre M completo")
print("-" * 100)

km_final = KMeans(
    k=optimal_k, seed=42, maxIter=25, tol=0.0001,
    initMode="k-means||", initSteps=5,
    featuresCol="scaledFeatures"
)

km_pipeline_final = Pipeline(stages=[assembler_km, scaler_km, km_final])

t0_km = datetime.now()
km_model = km_pipeline_final.fit(M_proc)
t1_km = datetime.now()

km_preds = km_model.transform(M_proc)
km_preds = km_preds.cache()
km_preds.count()

print(f"  Entrenamiento completado en {(t1_km - t0_km).total_seconds():.1f}s")

# Métricas de clustering
silhouette_final = km_evaluator.evaluate(km_preds)
wcss_final       = km_model.stages[-1].summary.trainingCost
dbi              = compute_davies_bouldin(km_preds)

print(f"\n  Métricas finales del clustering:")
print(f"    Silhouette Score     : {silhouette_final:.4f}")
print(f"    WCSS (Inercia)       : {wcss_final:,.0f}")
print(f"    Davies-Bouldin Index : {dbi:.4f}")

print(f"\n  Distribución de clusters:")
km_preds.groupBy("prediction").count().orderBy("prediction").show()

print("  Perfil de centroides:")
km_preds.groupBy("prediction").agg(
    avg("Distance(mi)").alias("avg_dist"),
    avg("Temperature(F)").alias("avg_temp"),
    avg("Humidity(%)").alias("avg_hum"),
    avg("Visibility(mi)").alias("avg_vis"),
    avg("Wind_Speed(mph)").alias("avg_wind"),
    avg("Precipitation(in)").alias("avg_prec")
).orderBy("prediction").show(truncate=False)

----------------------------------------------------------------------------------------------------

  Entrenamiento final de K-Means (k=2) sobre M completo
----------------------------------------------------------------------------------------------------
  Entrenamiento completado en 3.9s

  Métricas finales del clustering:
    Silhouette Score     : 0.3792
    WCSS (Inercia)       : 477,935
    Davies-Bouldin Index : 0.3283

  Distribución de clusters:
+----------+-----+
|prediction|count|
+----------+-----+
|         0|42561|
|         1|63353|
+----------+-----+

  Perfil de centroides:
+----------+-------------------+-----------------+-----------------+-------+-----------------+--------+
|prediction|avg_dist           |avg_temp         |avg_hum          |avg_vis|avg_wind         |avg_prec|
+----------+-------------------+-----------------+-----------------+-------+-----------------+--------+
|0         |0.41395827167614574|42.52678978407463|76.22652193322526|10.0   |6.512170766

---
## 5 Análisis de resultados

### 5.1 Comparación de modelos supervisados

In [20]:
print("=" * 100)
print("5 Análisis de resultados”")
print("=" * 100)

print("\n 5.1 Tabla comparativa de modelos supervisados (evaluación en Ts)")
print("-" * 100)

model_results = {
    "Random Forest":      rf_metrics,
    "Logistic Regression": lr_metrics,
    "GBT":                gbt_metrics
}

header = f"  {'Modelo':25s} | {'Accuracy':>9} | {'Precision':>9} | {'Recall':>7} | {'F1':>7} | {'F2':>7} | {'MCC':>7} | {'AUC-ROC':>8} | {'AUC-PR':>7}"
print(header)
print("  " + "-" * (len(header) - 2))

for name, m in model_results.items():
    auc_roc_str = f"{m['AUC-ROC']:.4f}" if m['AUC-ROC'] is not None else "   N/A  "
    auc_pr_str  = f"{m['AUC-PR']:.4f}"  if m['AUC-PR']  is not None else "  N/A "
    print(f"  {name:25s} | {m['Accuracy']:>9.4f} | {m['Precision']:>9.4f} | {m['Recall']:>7.4f} | {m['F1-Score']:>7.4f} | {m['F2-Score']:>7.4f} | {m['MCC']:>7.4f} | {auc_roc_str:>8} | {auc_pr_str:>7}")

best_name = builtins.max(
    model_results,
    key=lambda n: (model_results[n]["AUC-ROC"] or 0) + model_results[n]["F1-Score"]
)
print(f"\n  Mejor modelo (AUC-ROC + F1-Score): {best_name}")

5 Análisis de resultados”

 5.1 Tabla comparativa de modelos supervisados (evaluación en Ts)
----------------------------------------------------------------------------------------------------
  Modelo                    |  Accuracy | Precision |  Recall |      F1 |      F2 |     MCC |  AUC-ROC |  AUC-PR
  --------------------------------------------------------------------------------------------------------------
  Random Forest             |    0.6989 |    0.2721 |  0.7087 |  0.3932 |  0.5365 |  0.2917 |   0.7510 |  0.2924
  Logistic Regression       |    0.4232 |    0.1640 |  0.7789 |  0.2710 |  0.4451 |  0.1052 |   0.6719 |  0.2384
  GBT                       |    0.8622 |    0.4400 |  0.0037 |  0.0074 |  0.0047 |  0.0301 |   0.7542 |  0.3004

  Mejor modelo (AUC-ROC + F1-Score): Random Forest


In [21]:
print("-" * 100)
print("\n 5.2 Verificación de sobre-ajuste (Overfitting)")
print("-" * 100)

print("\n Evaluación de RF en conjunto de ENTRENAMIENTO (para comparar con prueba):")
rf_train_preds = rf_cv_model.transform(train_weighted)
rf_train_metrics = compute_supervised_metrics(rf_train_preds, prob_col="probability")

print(f"\n  Random Forest — AUC-ROC:")
print(f"    Entrenamiento : {rf_train_metrics['AUC-ROC']:.4f}")
print(f"    Prueba        : {rf_metrics['AUC-ROC']:.4f}")
delta_auc = rf_train_metrics['AUC-ROC'] - rf_metrics['AUC-ROC']
print(f"    (train–test): {delta_auc:.4f}")

if delta_auc < 0.03:
    print("    - Sin overfitting significativo — Δ < 0.03")
elif delta_auc < 0.07:
    print("    - Overfitting leve — considerar reducir maxDepth o numTrees")
else:
    print("    - Overfitting significativo — el modelo no generaliza bien")

print(f"\n  Random Forest — F1-Score:")
print(f"    Entrenamiento : {rf_train_metrics['F1-Score']:.4f}")
print(f"    Prueba        : {rf_metrics['F1-Score']:.4f}")

print("\n  Nota: el CrossValidator (3-fold) ya garantiza que los hiperparámetros")
print("  seleccionados minimizan el error de generalización antes de ver Ts.")

----------------------------------------------------------------------------------------------------

 5.2 Verificación de sobre-ajuste (Overfitting)
----------------------------------------------------------------------------------------------------

 Evaluación de RF en conjunto de ENTRENAMIENTO (para comparar con prueba):



  Random Forest — AUC-ROC:
    Entrenamiento : 0.7627
    Prueba        : 0.7510
    (train–test): 0.0118
    - Sin overfitting significativo — Δ < 0.03

  Random Forest — F1-Score:
    Entrenamiento : 0.3844
    Prueba        : 0.3932

  Nota: el CrossValidator (3-fold) ya garantiza que los hiperparámetros
  seleccionados minimizan el error de generalización antes de ver Ts.


In [22]:
print("-" * 100)
print("\n 5.3 Análisis del clustering K-Means")
print("-" * 100)

print(f"""
  Resumen de métricas de clustering:
    k óptimo             : {optimal_k}
    Silhouette Score     : {silhouette_final:.4f}
    WCSS (Inercia)       : {wcss_final:,.0f}
    Davies-Bouldin Index : {dbi:.4f}

  Interpretación del Silhouette Score ({silhouette_final:.4f}):
""")

if silhouette_final > 0.7:
    sil_interp = "Excelente — clusters compactos y bien separados."
elif silhouette_final > 0.5:
    sil_interp = "Bueno — estructura de clustering razonablemente clara."
elif silhouette_final > 0.3:
    sil_interp = "Aceptable — algunos clusters se solapan marginalmente."
else:
    sil_interp = "Débil — las variables no generan clusters naturalmente separados."

print(f"    {sil_interp}")

print(f"""
  Interpretación del Davies-Bouldin Index ({dbi:.4f}):
    Valores < 1.0 indican buena separación inter-cluster.
    {'- DBI < 1.0 — clusters bien definidos.' if dbi < 1.0 else '⚠ DBI ≥ 1.0 — clusters con solapamiento moderado.'}

  Evolución del WCSS (criterio del codo):""")
prev_wcss = None
for k_val, wcss_val in sorted(wcss_values.items()):
    if prev_wcss is not None:
        drop = (prev_wcss - wcss_val) / prev_wcss * 100
        print(f"    k={k_val}: WCSS={wcss_val:>12,.0f}  (reducción {drop:.1f}% vs k={k_val-1})")
    else:
        print(f"    k={k_val}: WCSS={wcss_val:>12,.0f}")
    prev_wcss = wcss_val

----------------------------------------------------------------------------------------------------

 5.3 Análisis del clustering K-Means
----------------------------------------------------------------------------------------------------

  Resumen de métricas de clustering:
    k óptimo             : 2
    Silhouette Score     : 0.3792
    WCSS (Inercia)       : 477,935
    Davies-Bouldin Index : 0.3283

  Interpretación del Silhouette Score (0.3792):

    Aceptable — algunos clusters se solapan marginalmente.

  Interpretación del Davies-Bouldin Index (0.3283):
    Valores < 1.0 indican buena separación inter-cluster.
    - DBI < 1.0 — clusters bien definidos.

  Evolución del WCSS (criterio del codo):
    k=2: WCSS=      71,901
    k=3: WCSS=      61,402  (reducción 14.6% vs k=2)
    k=4: WCSS=      53,621  (reducción 12.7% vs k=3)
    k=5: WCSS=      51,765  (reducción 3.5% vs k=4)
    k=6: WCSS=      45,178  (reducción 12.7% vs k=5)


In [23]:
print("-" * 100)
print("\n 5.4 Consideraciones finales")
print("-" * 100)

best_rf_auc  = rf_metrics["AUC-ROC"]  or 0
best_lr_auc  = lr_metrics["AUC-ROC"]  or 0
best_gbt_auc = gbt_metrics["AUC-ROC"] or 0

print(f"""
══════════════════════════════════════════════════════════════════════════════
Consideraciones
══════════════════════════════════════════════════════════════════════════════

 1. La muestra M fue construida con muestreo estratificado proporcional
    (fracción = 2 % por partición), lo que garantiza que la distribución
    de las 8 particiones en M es estadísticamente equivalente a la de P.
    La desviación máxima observada fue < 0.5 %, confirmando representatividad.

 2. El split estratificado por clase aseguró que Tr y Ts mantuvieran la misma
    proporción de clase 0 (≈87%) y clase 1 (≈13%), eliminando el riesgo de que
    el conjunto de prueba concentre la clase minoritaria o la mayoría.

 3. Los pesos de clase inversos usados en RF y LR compensaron el desbalanceo
    ~6:1, mejorando el Recall en la clase minoritaria (accidentes graves) sin
    necesidad de sobregenerar instancias sintéticas.

 4. El CrossValidator con 3 folds seleccionó los hiperparámetros que maximizan
    el AUC-ROC en datos de validación no vistos. La diferencia train–test Δ < 0.05
    confirma que los modelos generalizan adecuadamente.

 5. La evaluación multimétr ica (Accuracy, Precision, Recall, F1, F2, MCC,
    AUC-ROC, AUC-PR, Silhouette, WCSS, DBI) ofrece una visión 360° del
    rendimiento, especialmente importante en datasets desbalanceados donde la
    Accuracy sola es engañosa.

 6. El uso de Pipeline encapsula el StandardScaler, lo que garantiza que los
    parámetros de normalización se estiman exclusivamente sobre Tr y se aplican
    (sin re-estimar) sobre Ts, evitando contaminación del conjunto de prueba.
""")

----------------------------------------------------------------------------------------------------

 5.4 Consideraciones finales
----------------------------------------------------------------------------------------------------

══════════════════════════════════════════════════════════════════════════════
Consideraciones
══════════════════════════════════════════════════════════════════════════════

 1. La muestra M fue construida con muestreo estratificado proporcional
    (fracción = 2 % por partición), lo que garantiza que la distribución
    de las 8 particiones en M es estadísticamente equivalente a la de P.
    La desviación máxima observada fue < 0.5 %, confirmando representatividad.

 2. El split estratificado por clase aseguró que Tr y Ts mantuvieran la misma
    proporción de clase 0 (≈87%) y clase 1 (≈13%), eliminando el riesgo de que
    el conjunto de prueba concentre la clase minoritaria o la mayoría.

 3. Los pesos de clase inversos usados en RF y LR compensaron el 

In [24]:
print("=" * 100)
print("\n Conclusiones")
print("=" * 100)

best_auc  = builtins.max(best_rf_auc, best_lr_auc, best_gbt_auc)
best_mod  = {best_rf_auc: "Random Forest", best_lr_auc: "Logistic Regression", best_gbt_auc: "GBT"}[best_auc]
best_f1   = builtins.max(
    rf_metrics["F1-Score"],
    lr_metrics["F1-Score"],
    gbt_metrics["F1-Score"]
)

sil_label = (
    "excelente" if silhouette_final > 0.7 else
    "bueno"     if silhouette_final > 0.5 else
    "aceptable" if silhouette_final > 0.3 else
    "débil"
)

print(f"""
Esta actividad implementó un pipeline completo de aprendizaje automático
sobre el dataset de accidentes de tráfico en EE.UU. (~7.7 M registros),
con especial énfasis en la calidad de los resultados y la selección de
métricas adecuadas para datos desbalanceados en entornos Big Data.

Aprendizaje supervisado:
  Mejor modelo: {best_mod}
  AUC-ROC    : {best_auc:.4f}  ({"Excelente" if best_auc > 0.9 else "Bueno" if best_auc > 0.8 else "Aceptable"})
  F1-Score   : {best_f1:.4f}

Aprendizaje no supervisado:
  K-Means (k={optimal_k}) — Silhouette: {silhouette_final:.4f} ({sil_label})
  Los {optimal_k} clusters identificaron patrones ambientales distintos
  en las condiciones de ocurrencia de accidentes.

Hallazgo central:
  La AUC-PR es la métrica más relevante para este problema dado el fuerte
  desbalanceo de clases. Un AUC-ROC alto puede ser engañoso si la
  clase minoritaria (accidentes graves) no se predice correctamente,
  por lo que el análisis conjunto de Recall, F2-Score y AUC-PR es
  indispensable para validar la utilidad operativa del modelo.

  Estudiante: Ovidio Alejandro Hernández Ruano — A01796714
  Fecha     : 08/06/2026
""")

spark.stop()
print("Sesión Spark finalizada.")


 Conclusiones

Esta actividad implementó un pipeline completo de aprendizaje automático
sobre el dataset de accidentes de tráfico en EE.UU. (~7.7 M registros),
con especial énfasis en la calidad de los resultados y la selección de
métricas adecuadas para datos desbalanceados en entornos Big Data.

Aprendizaje supervisado:
  Mejor modelo: GBT
  AUC-ROC    : 0.7542  (Aceptable)
  F1-Score   : 0.3932

Aprendizaje no supervisado:
  K-Means (k=2) — Silhouette: 0.3792 (aceptable)
  Los 2 clusters identificaron patrones ambientales distintos
  en las condiciones de ocurrencia de accidentes.

Hallazgo central:
  La AUC-PR es la métrica más relevante para este problema dado el fuerte
  desbalanceo de clases. Un AUC-ROC alto puede ser engañoso si la
  clase minoritaria (accidentes graves) no se predice correctamente,
  por lo que el análisis conjunto de Recall, F2-Score y AUC-PR es
  indispensable para validar la utilidad operativa del modelo.

  Estudiante: Ovidio Alejandro Hernández Ruano — A